In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Tugas12_KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

df = spark.read.csv(
    "tugas12_ulasan.csv",
    header=True,
    inferSchema=True
)

df.printSchema()
df.show(5)

root
 |-- nama_menu: string (nullable = true)
 |-- cabang: string (nullable = true)
 |-- waktu_pesan: string (nullable = true)
 |-- qty: integer (nullable = true)
 |-- waktu_tunggu_menit: integer (nullable = true)
 |-- ulasan_positif: integer (nullable = true)

+--------------+----------+-----------+---+------------------+--------------+
|     nama_menu|    cabang|waktu_pesan|qty|waktu_tunggu_menit|ulasan_positif|
+--------------+----------+-----------+---+------------------+--------------+
|     Kopi Susu|Yogyakarta|       Sore|  5|                 5|             1|
|        Es Teh|Yogyakarta|       Pagi|  1|                21|             0|
|    Cappuccino|Yogyakarta|       Sore|  5|                15|             0|
|Kentang Goreng|  Semarang|      Malam|  2|                 5|             1|
|     Americano|Yogyakarta|      Malam|  2|                10|             0|
+--------------+----------+-----------+---+------------------+--------------+
only showing top 5 rows



**A. Membangun ML Pipeline**

In [4]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression

idx_menu = StringIndexer(inputCol="nama_menu", outputCol="menu_idx", handleInvalid="keep")
idx_cabang = StringIndexer(inputCol="cabang", outputCol="cabang_idx", handleInvalid="keep")
idx_waktu = StringIndexer(inputCol="waktu_pesan", outputCol="waktu_idx", handleInvalid="keep")

ohe = OneHotEncoder(
    inputCols=["menu_idx", "cabang_idx", "waktu_idx"],
    outputCols=["menu_vec", "cabang_vec", "waktu_vec"]
)

assembler = VectorAssembler(
    inputCols=["qty", "waktu_tunggu_menit", "menu_vec", "cabang_vec", "waktu_vec"],
    outputCol="features"
)

lr = LogisticRegression(
    featuresCol="features",
    labelCol="ulasan_positif",
    maxIter=20
)

stages_lr = [idx_menu, idx_cabang, idx_waktu, ohe, assembler, lr]
pipeline_lr = Pipeline(stages=stages_lr)

print("Pipeline Logistic Regression berhasil dibuat!")
print(f"Jumlah tahapan dalam pipeline: {len(pipeline_lr.getStages())}")
for i, stage in enumerate(pipeline_lr.getStages(), start=1):
    print(f"Stage {i}: {type(stage).__name__}")

Pipeline Logistic Regression berhasil dibuat!
Jumlah tahapan dalam pipeline: 6
Stage 1: StringIndexer
Stage 2: StringIndexer
Stage 3: StringIndexer
Stage 4: OneHotEncoder
Stage 5: VectorAssembler
Stage 6: LogisticRegression


**B. Pelatihan & Evaluasi**

In [5]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

train_df, test_df = df.randomSplit([0.8, 0.2], seed=77)

model_lr = pipeline_lr.fit(train_df)
pred_lr = model_lr.transform(test_df)

eval_acc = MulticlassClassificationEvaluator(
    labelCol="ulasan_positif",
    predictionCol="prediction",
    metricName="accuracy"
)
eval_f1 = MulticlassClassificationEvaluator(
    labelCol="ulasan_positif",
    predictionCol="prediction",
    metricName="f1"
)
eval_auc = BinaryClassificationEvaluator(
    labelCol="ulasan_positif",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

acc_lr = eval_acc.evaluate(pred_lr)
f1_lr = eval_f1.evaluate(pred_lr)
auc_lr = eval_auc.evaluate(pred_lr)

print("=== Evaluasi Logistic Regression ===")
print(f"Accuracy : {acc_lr:.4f}")
print(f"F1-Score : {f1_lr:.4f}")
print(f"AUC-ROC  : {auc_lr:.4f}")

print("\n=== Confusion Matrix (Logistic Regression) ===")
pred_lr.groupBy("ulasan_positif", "prediction").count().orderBy("ulasan_positif", "prediction").show()

=== Evaluasi Logistic Regression ===
Accuracy : 0.7929
F1-Score : 0.7926
AUC-ROC  : 0.8650

=== Confusion Matrix (Logistic Regression) ===


[Stage 37:>                                                         (0 + 1) / 1]

+--------------+----------+-----+
|ulasan_positif|prediction|count|
+--------------+----------+-----+
|             0|       0.0|  257|
|             0|       1.0|   57|
|             1|       0.0|   65|
|             1|       1.0|  210|
+--------------+----------+-----+



**C. Model Pembanding**

In [6]:
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="ulasan_positif",
    numTrees=30,
    seed=77
)

stages_rf = [idx_menu, idx_cabang, idx_waktu, ohe, assembler, rf]
pipeline_rf = Pipeline(stages=stages_rf)

model_rf = pipeline_rf.fit(train_df)
pred_rf = model_rf.transform(test_df)

acc_rf = eval_acc.evaluate(pred_rf)
f1_rf = eval_f1.evaluate(pred_rf)
auc_rf = eval_auc.evaluate(pred_rf)

print("=== Evaluasi Random Forest (numTrees=30) ===")
print(f"Accuracy : {acc_rf:.4f}")
print(f"F1-Score : {f1_rf:.4f}")
print(f"AUC-ROC  : {auc_rf:.4f}")

print("\n=== Perbandingan Singkat ===")
print(f"Logistic Regression -> Acc: {acc_lr:.4f}, F1: {f1_lr:.4f}, AUC: {auc_lr:.4f}")
print(f"Random Forest       -> Acc: {acc_rf:.4f}, F1: {f1_rf:.4f}, AUC: {auc_rf:.4f}")

=== Evaluasi Random Forest (numTrees=30) ===
Accuracy : 0.7929
F1-Score : 0.7926
AUC-ROC  : 0.8467

=== Perbandingan Singkat ===
Logistic Regression -> Acc: 0.7929, F1: 0.7926, AUC: 0.8650
Random Forest       -> Acc: 0.7929, F1: 0.7926, AUC: 0.8467


**D. Analisis Kesalahan Model**

In [7]:
from pyspark.sql.functions import col

if auc_rf >= auc_lr:
    best_model_name = "Random Forest"
    best_pred = pred_rf
else:
    best_model_name = "Logistic Regression"
    best_pred = pred_lr

print(f"Model terbaik berdasarkan AUC-ROC: {best_model_name}")

salah_pred = best_pred.filter(col("ulasan_positif") != col("prediction"))

print("\n=== 10 Baris Salah Prediksi ===")
salah_pred.select(
    "nama_menu",
    "cabang",
    "waktu_pesan",
    "qty",
    "waktu_tunggu_menit",
    "ulasan_positif",
    "prediction"
).show(10, truncate=False)

print("\n=== Statistik waktu_tunggu_menit pada Salah Prediksi ===")
salah_pred.select("waktu_tunggu_menit").describe().show()

Model terbaik berdasarkan AUC-ROC: Logistic Regression

=== 10 Baris Salah Prediksi ===
+---------+--------+-----------+---+------------------+--------------+----------+
|nama_menu|cabang  |waktu_pesan|qty|waktu_tunggu_menit|ulasan_positif|prediction|
+---------+--------+-----------+---+------------------+--------------+----------+
|Americano|Magelang|Malam      |1  |15                |1             |0.0       |
|Americano|Magelang|Siang      |2  |22                |1             |0.0       |
|Americano|Magelang|Sore       |1  |7                 |0             |1.0       |
|Americano|Magelang|Sore       |2  |11                |0             |1.0       |
|Americano|Magelang|Sore       |3  |5                 |0             |1.0       |
|Americano|Magelang|Sore       |3  |13                |1             |0.0       |
|Americano|Magelang|Sore       |5  |6                 |0             |1.0       |
|Americano|Magelang|Sore       |5  |15                |1             |0.0       |
|Americano

**E. Kesimpulan Bisnis**

Berdasarkan hasil pemodelan dan analisis kesalahan pada data Kopi Nusantara, fitur yang paling berpengaruh terhadap prediksi ulasan positif adalah waktu_tunggu_menit, di mana semakin lama pelanggan menunggu pesanan maka peluang ulasan positif cenderung menurun. Pada bagian analisis kesalahan, kekeliruan prediksi model banyak muncul pada kasus dengan waktu_tunggu_menit yang berada di sekitar area ambang batas atau transisi, ketika respons pelanggan bersifat fluktuatif meski waktu tunggunya relatif mirip. Oleh karena itu, manajemen Kopi Nusantara disarankan menetapkan target batas waktu tunggu ideal (misalnya menjaga waktu tunggu di bawah kisaran 10–12 menit) serta memantau antrean pada jam sibuk (waktu_pesan) dan cabang tertentu (cabang). Selain itu, manajemen perlu mengoptimalkan alur penyiapan menu yang membutuhkan waktu proses lebih lama atau memiliki pesanan berjumlah besar (qty) agar kepuasan pelanggan tetap terjaga secara konsisten di seluruh cabang.

**EKSPLORASI**

In [9]:
from pyspark.sql.functions import avg, count, round

print("PERFORMA PER CABANG")
df.groupBy("cabang") \
  .agg(
      count("*").alias("total_order"),
      round(avg("ulasan_positif") * 100, 2).alias("persen_ulasan_positif"),
      round(avg("waktu_tunggu_menit"), 2).alias("rata_tunggu_menit")
  ) \
  .orderBy("cabang") \
  .show(truncate=False)

PERFORMA PER CABANG
+----------+-----------+---------------------+-----------------+
|cabang    |total_order|persen_ulasan_positif|rata_tunggu_menit|
+----------+-----------+---------------------+-----------------+
|Magelang  |946        |45.88                |13.26            |
|Semarang  |968        |45.25                |13.03            |
|Yogyakarta|1086       |47.7                 |13.11            |
+----------+-----------+---------------------+-----------------+



In [10]:
from pyspark.sql.functions import avg, count, round, col

print("PERFORMA PER MENU")
df.groupBy("nama_menu") \
  .agg(
      count("*").alias("total_order"),
      round(avg("qty"), 2).alias("rata_qty"),
      round(avg("waktu_tunggu_menit"), 2).alias("rata_tunggu_menit"),
      round(avg("ulasan_positif") * 100, 2).alias("persen_ulasan_positif")
  ) \
  .orderBy(col("persen_ulasan_positif").desc()) \
  .show(truncate=False)

PERFORMA PER MENU
+--------------+-----------+--------+-----------------+---------------------+
|nama_menu     |total_order|rata_qty|rata_tunggu_menit|persen_ulasan_positif|
+--------------+-----------+--------+-----------------+---------------------+
|Kopi Susu     |544        |2.94    |13.18            |48.71                |
|Americano     |502        |2.95    |12.55            |47.81                |
|Cappuccino    |503        |2.99    |13.06            |47.71                |
|Kentang Goreng|464        |2.99    |13.04            |46.98                |
|Es Teh        |486        |3.04    |13.75            |43.62                |
|Roti Bakar    |501        |2.95    |13.23            |42.91                |
+--------------+-----------+--------+-----------------+---------------------+



In [11]:
from pyspark.sql.functions import avg, count, round

print("PENGARUH WAKTU PESAN")
df.groupBy("waktu_pesan") \
  .agg(
      count("*").alias("total_order"),
      round(avg("waktu_tunggu_menit"), 2).alias("rata_tunggu_menit"),
      round(avg("ulasan_positif") * 100, 2).alias("persen_ulasan_positif")
  ) \
  .orderBy("waktu_pesan") \
  .show(truncate=False)

PENGARUH WAKTU PESAN
+-----------+-----------+-----------------+---------------------+
|waktu_pesan|total_order|rata_tunggu_menit|persen_ulasan_positif|
+-----------+-----------+-----------------+---------------------+
|Malam      |732        |13.22            |47.81                |
|Pagi       |772        |12.99            |44.95                |
|Siang      |752        |13.23            |45.48                |
|Sore       |744        |13.09            |47.18                |
+-----------+-----------+-----------------+---------------------+



In [12]:
from pyspark.sql.functions import avg, count, round, when, col

print("KELOMPOK WAKTU TUNGGU")
df_kelompok = df.withColumn(
    "kategori_tunggu",
    when(col("waktu_tunggu_menit") <= 10, "Cepat (<=10 mnt)")
    .when(col("waktu_tunggu_menit") <= 15, "Sedang (11-15 mnt)")
    .otherwise("Lama (>15 mnt)")
)

df_kelompok.groupBy("kategori_tunggu") \
  .agg(
      count("*").alias("jumlah_data"),
      round(avg("ulasan_positif") * 100, 2).alias("persen_ulasan_positif")
  ) \
  .orderBy("kategori_tunggu") \
  .show(truncate=False)

KELOMPOK WAKTU TUNGGU
+------------------+-----------+---------------------+
|kategori_tunggu   |jumlah_data|persen_ulasan_positif|
+------------------+-----------+---------------------+
|Cepat (<=10 mnt)  |1162       |79.52                |
|Lama (>15 mnt)    |1200       |15.92                |
|Sedang (11-15 mnt)|638        |43.1                 |
+------------------+-----------+---------------------+



In [13]:
spark.stop()
print("SparkSession berhasil dihentikan.")

SparkSession berhasil dihentikan.
